# Week 19 (AWS DI Variant): MLOps for Data Engineers - Versioning, Quality Gates, and Pipeline Lineage

The standard Week 19 notebook is written for ML engineers who fine-tune a model and push it to production. This variant is for the data engineer who has to GUARANTEE that the data feeding that model is versioned, validated, and traceable - on AWS SageMaker.

You will not train a model in this notebook. You will produce the artifact a model training job consumes: a frozen, quality-checked, lineage-tagged training-ready Parquet snapshot in a versioned S3 prefix, with a parent MLflow run that records exactly which S3 VersionId, which validation checks passed, and which feature transformations were applied.

## Learning objectives

By the end of this session you will be able to:

1. Pin an S3 object read to a specific `VersionId` using `list_object_versions` + `get_object(VersionId=...)` and explain why this is what reproducibility actually means.
2. Write pandas data quality assertions (null rate, class balance, schema drift) and log pass/fail to MLflow as a data validation run.
3. Structure an MLflow experiment as a parent feature-pipeline run with a child training run, so lineage is queryable.
4. Write a versioned Parquet snapshot to a student-scoped S3 prefix (`s3://bucket/students/<id>/fraud_train/v<N>/`) using `pandas.to_parquet` + `boto3.upload_file`.
5. Register a SageMaker model package with `CustomerMetadataProperties` pointing at the parent feature pipeline run, the S3 VersionId, and the Parquet snapshot prefix.

## Prerequisites

- Week 14 (DistilBERT fine-tuning) - awareness only
- Week 18 (RAG pipeline, KB id FARSQGTONR) - awareness only; we keep continuity by reusing the same shared infra naming
- The standard Week 19 ML engineer notebook - recommended but not required

## Environment Setup

**Platform**: AWS SageMaker Studio (kernel: Data Science 3.0 or any Python 3 image with pip).

**Install** (run the next cell once per kernel):

- `sagemaker==2.257.3` (pin to v2; v3 breaks `from sagemaker import get_execution_role`)
- `boto3>=1.35`
- `awswrangler>=3.9` (pandas-on-S3, used for Parquet I/O)
- `mlflow>=2.13` (local file-store backend; no tracking server required)
- `pyarrow>=15` (Parquet engine for pandas)

**Auth**: the SageMaker execution role auto-resolves AWS credentials. No `getpass`, no access keys, no `dbutils`.

**Shared bucket prerequisites** (instructor verifies before class):
- `s3://bread-academy-week19-shared` exists with S3 versioning ENABLED
- `s3://bread-academy-week19-shared/raw/fraud_transactions.csv` uploaded (instructor uploads it TWICE in pre-class to create multiple versions, so Lab 1 has something to roll back to)
- `s3://bread-academy-week19-shared/pretrained/model.tar.gz` present (reused from the main Week 19 notebook)
- The student's SageMaker execution role has `s3:GetObject`, `s3:GetObjectVersion`, `s3:ListBucket`, `s3:ListBucketVersions`, `s3:PutObject` on the bucket

In [ ]:
# Install pinned libraries (idempotent; safe to re-run)
!pip install -q "sagemaker==2.257.3" "boto3>=1.35" "awswrangler>=3.9" "mlflow==3.10.0" "pyarrow>=15"

# Standard library
import os
import io
import json
import time
from datetime import datetime

# Third-party
import boto3
from botocore.exceptions import ClientError
import pandas as pd
import awswrangler as wr
import mlflow
from importlib.metadata import version

# Verify versions (no __version__ access - use importlib.metadata per house style)
for pkg in ["boto3", "sagemaker", "awswrangler", "mlflow", "pyarrow"]:
    try:
        print(f"{pkg:15s} {version(pkg)}")
    except Exception as e:
        print(f"{pkg:15s} NOT INSTALLED ({e})")

In [ ]:
import sagemaker
from sagemaker import get_execution_role

# Execution role auto-resolves. No keys, no MFA in code.
sess = sagemaker.Session()
role = get_execution_role()
AWS_REGION = sess.boto_region_name

# Export region so downstream libs (awswrangler, strands_tools, etc.) pick it up
os.environ["AWS_REGION"]         = AWS_REGION
os.environ["AWS_DEFAULT_REGION"] = AWS_REGION

# boto3 clients (credentials sourced from the execution role)
boto_session     = boto3.Session(region_name=AWS_REGION)
s3_client        = boto_session.client("s3")
sagemaker_client = boto_session.client("sagemaker")
sts_client       = boto_session.client("sts")

print(f"Region:        {AWS_REGION}")
print(f"Role:          {role}")
print(f"Caller:        {sts_client.get_caller_identity()['Arn']}")

In [ ]:
# Pre-flight probes - fail loud before any real work.
S3_BUCKET    = "bread-academy-week19-shared"
SOURCE_KEY   = "raw/fraud_transactions.csv"
STUDENT_ID   = sts_client.get_caller_identity()["UserId"][:8].lower()
S3_PREFIX    = f"students/{STUDENT_ID}"

# 1) Bucket reachable
try:
    s3_client.head_bucket(Bucket=S3_BUCKET)
    print(f"S3 bucket OK: s3://{S3_BUCKET}")
except Exception as e:
    print(f"S3 bucket FAIL: {e}\nAsk your instructor to grant access to {S3_BUCKET}.")
    raise

# 2) Versioning ENABLED on the bucket (this is the whole point of Topic 1)
vstate = s3_client.get_bucket_versioning(Bucket=S3_BUCKET).get("Status")
if vstate != "Enabled":
    raise RuntimeError(
        f"S3 versioning is {vstate} on {S3_BUCKET}. "
        "Ask your instructor to enable versioning (Topic 1 requires it)."
    )
print(f"S3 versioning OK: {vstate}")

# 3) Source CSV present
try:
    head = s3_client.head_object(Bucket=S3_BUCKET, Key=SOURCE_KEY)
    print(f"Source CSV OK: s3://{S3_BUCKET}/{SOURCE_KEY} (ETag={head['ETag']}, VersionId={head.get('VersionId')})")
except Exception as e:
    print(f"Source CSV FAIL: {e}\nAsk your instructor to upload {SOURCE_KEY}.")
    raise

# 4) SageMaker reachable (we will create model packages later)
try:
    sagemaker_client.list_training_jobs(MaxResults=1)
    print("SageMaker OK")
except Exception as e:
    print(f"SageMaker FAIL: {e}")
    raise

# 5) MLflow local file-store tracking
MLFLOW_DIR = os.path.abspath("./mlruns")
mlflow.set_tracking_uri(f"file://{MLFLOW_DIR}")
print(f"MLflow OK: tracking_uri={mlflow.get_tracking_uri()}")

## What Are We Building Today?

Picture the conversation that happens at Bread Financial on a Monday morning:

> ML engineer: "The fraud model got worse this week. Can you check the training data?"
> Data engineer: "Which week's data? Which version of the CSV in S3? Did anyone validate it before training?"
> ML engineer: "I don't know. I just pointed the job at the latest object in the bucket."

That is the gap this notebook closes. By the end you will have a hand-off contract that looks like this:

1. The **source CSV in S3** has a recorded `VersionId`, captured at the moment of training.
2. A **data validation run** in MLflow proves the CSV passed quality gates BEFORE training started.
3. A **feature pipeline parent run** in MLflow lists every transformation, the row count after each step, and the output schema.
4. A **child training run** under that parent loads the same Parquet snapshot the parent produced, and inherits the lineage tags.
5. The **model package** in the SageMaker Registry has `CustomerMetadataProperties` pointing back to the parent MLflow run, the source S3 VersionId, and the Parquet snapshot prefix.

When the ML engineer asks "what data trained this model?" you do not say "the latest CSV." You give them an MLflow run id and a 64-character S3 VersionId, and they can replay the entire lineage in 30 seconds.

This continues the thread from Week 18: that RAG pipeline (KB `FARSQGTONR`, Cohere reranker, Haiku 3 LLM) consumed documents that nobody version-pinned. The same upstream discipline you build today applies there too.

## Topic 1: S3 object versioning for ML reproducibility

When versioning is enabled on an S3 bucket, every `PutObject` creates a new version of that key with a unique `VersionId` (a long opaque string). Older versions are kept until you explicitly delete them, and you can read any historical version by passing `VersionId` to `get_object`.

For an ML training pipeline, this means **"the data version" is a real, queryable thing**, not a Slack message saying "I think it was Tuesday's upload."

Three operations matter:

- `s3.list_object_versions(Bucket=..., Prefix=...)` - lists every historical `VersionId` of every object under the prefix, newest first.
- `s3.get_object(Bucket=..., Key=..., VersionId=<v>)` - reads the object exactly as it existed at that version.
- `s3.head_object(Bucket=..., Key=...)` - returns the CURRENT VersionId (the one you would read by default).

This is the SageMaker-native equivalent of Delta Lake time travel. It is simpler (one bucket-level flag, no transaction log) but has the same reproducibility property: as long as the version is not lifecycle-expired, you can replay the exact bytes the training job consumed.

In production, you protect long-term reproducibility two ways:
- **Don't lifecycle-expire training versions**: tag the VersionIds you trained on, and exclude tagged versions from the bucket's lifecycle rule.
- **Or copy the trained-on bytes to a versioned snapshot prefix you fully control** (Topic 4 does this).

In [ ]:
# Inspect every historical version of the source CSV
versions_resp = s3_client.list_object_versions(Bucket=S3_BUCKET, Prefix=SOURCE_KEY)
all_versions = [
    v for v in versions_resp.get("Versions", [])
    if v["Key"] == SOURCE_KEY
]
# S3 returns newest first
versions_df = pd.DataFrame([{
    "VersionId":     v["VersionId"],
    "LastModified":  v["LastModified"],
    "Size":          v["Size"],
    "IsLatest":      v["IsLatest"],
} for v in all_versions])
print(versions_df.head(10))

# Capture the current (latest) version. We pin every read in this notebook to this id.
latest = next(v for v in all_versions if v["IsLatest"])
DATA_VERSION_ID = latest["VersionId"]
DATA_TIMESTAMP  = str(latest["LastModified"])
print(f"\nPinned DATA_VERSION_ID = {DATA_VERSION_ID} (written at {DATA_TIMESTAMP})")

In [ ]:
# Two equivalent ways to read an S3 object pinned to a specific version.

# Way 1: raw boto3 get_object(VersionId=...) into a pandas DataFrame via BytesIO
obj = s3_client.get_object(Bucket=S3_BUCKET, Key=SOURCE_KEY, VersionId=DATA_VERSION_ID)
boto_df = pd.read_csv(io.BytesIO(obj["Body"].read()))
print(f"boto3 pinned read: {len(boto_df)} rows, columns={list(boto_df.columns)}")

# Way 2: awswrangler (pandas-on-S3) - cleaner for production DE code
# awswrangler supports version_id via the s3 client kwargs
aws_df = wr.s3.read_csv(
    path=f"s3://{S3_BUCKET}/{SOURCE_KEY}",
    version_id=DATA_VERSION_ID,
    boto3_session=boto_session,
)
print(f"awswrangler pinned read: {len(aws_df)} rows")

# Sanity: both reads should produce identical row counts and identical bytes-of-content
assert len(boto_df) == len(aws_df), "Pinned reads disagree on row count"
assert sorted(boto_df.columns) == sorted(aws_df.columns), "Pinned reads disagree on schema"
print("OK: both pinned reads agree.")

# Keep one canonical DataFrame for downstream cells
data_df = aws_df.copy()

### Lab 1: Reproduce a read at an older version

You are going to simulate the question "what did the CSV look like before the most recent upload?"

**Steps**:

1. From `all_versions`, pick the SECOND entry (the previous version of the same key). Store its `VersionId` as `previous_version_id`.
2. Read the source CSV pinned to that older `VersionId` into a DataFrame named `older_df`. You can use either `boto3.get_object` or `awswrangler.s3.read_csv`.
3. Print the row count of `older_df` and compare to the latest version's row count.
4. Store the difference as `row_delta` (latest minus older). A positive number means rows were added; negative means rows were removed.

**Stretch**: Compare not just row counts but also a hash of the sorted DataFrame content (e.g. `pd.util.hash_pandas_object(df).sum()`). If the row counts match but the hash differs, the upstream owner mutated existing rows in place.

**Homework Extension**: Read the AWS docs on S3 Lifecycle rules and noncurrent version expiration. Write 3 sentences explaining why "VersionId from a year ago" is NOT a reliable reproducibility strategy on its own, and what you would do instead (hint: tag the VersionIds you trained on, or snapshot to a separate prefix).

In [ ]:
# Lab 1: read the previous version of the source CSV

previous_version_id = None  # YOUR CODE
older_df = None  # YOUR CODE
row_delta = None  # YOUR CODE

# SAFETY-NET for Lab 1 - run this if you didn't finish. SKIP if you completed it.
if previous_version_id is None:
    print("Using Lab 1 safety-net.")
    if len(all_versions) < 2:
        raise RuntimeError(
            "Only one version of the source CSV exists. "
            "Ask your instructor to re-upload it so there is a previous version."
        )
    previous_version_id = all_versions[1]["VersionId"]
    older_obj = s3_client.get_object(Bucket=S3_BUCKET, Key=SOURCE_KEY, VersionId=previous_version_id)
    older_df  = pd.read_csv(io.BytesIO(older_obj["Body"].read()))
    row_delta = len(data_df) - len(older_df)
    print(f"previous_version_id={previous_version_id}")
    print(f"older rows={len(older_df)}, latest rows={len(data_df)}, delta={row_delta}")

## Topic 2: Data quality gates before retraining

A training job that runs on bad data produces a bad model on schedule. The data engineer's job is to stop bad data BEFORE the training job consumes it.

In a real Bread Financial pipeline you might use Great Expectations, Soda, or AWS Glue Data Quality for this. For class we will use plain pandas assertions because they are easier to read and require no extra infra. The pattern is identical:

1. Define a list of checks (null rate, class balance, expected schema).
2. Run each check against the pinned data.
3. Collect pass/fail + measured value for each check.
4. Log the whole batch to MLflow as a "data validation run". If any HARD check fails, raise an exception so the training job never starts.

The MLflow run gives you an audit trail: every S3 VersionId that ever fed a training job has a paired validation run you can point an auditor at.

Note on the AWS-native option: AWS Glue Data Quality (DQDL) is the SageMaker-native production choice. It runs as a managed Spark job and integrates with Data Catalog. We skip it here because spinning it up costs 5+ minutes of class time per check; the inline pandas pattern is the same idea and runs in the kernel.

In [ ]:
EXPECTED_SCHEMA = {"narrative": "object", "is_fraud": "int64"}  # pandas dtype names
MAX_NULL_RATE   = 0.01      # at most 1 percent nulls in label column
MIN_FRAUD_RATIO = 0.02      # at least 2 percent positive class
MAX_FRAUD_RATIO = 0.50      # at most 50 percent positive class

def run_checks(df: pd.DataFrame) -> dict:
    results = {}

    # 1. Schema check (subset of columns we care about)
    actual = {c: str(df[c].dtype) for c in df.columns}
    for col_name, expected_type in EXPECTED_SCHEMA.items():
        got = actual.get(col_name)
        results[f"schema_{col_name}"] = {
            "passed": got is not None and got.startswith(expected_type),
            "value":  str(got),
            "hard":   True,
        }

    # 2. Null rate on label column
    total       = len(df)
    null_labels = int(df["is_fraud"].isna().sum()) if "is_fraud" in df.columns else total
    null_rate   = null_labels / total if total > 0 else 1.0
    results["null_rate_label"] = {"passed": null_rate <= MAX_NULL_RATE, "value": null_rate, "hard": True}

    # 3. Class balance
    fraud_count = int((df["is_fraud"] == 1).sum()) if "is_fraud" in df.columns else 0
    fraud_ratio = fraud_count / total if total > 0 else 0.0
    results["fraud_ratio"] = {
        "passed": MIN_FRAUD_RATIO <= fraud_ratio <= MAX_FRAUD_RATIO,
        "value":  fraud_ratio,
        "hard":   False,
    }

    # 4. Row count floor
    results["row_count"] = {"passed": total >= 200, "value": total, "hard": True}

    return results

check_results = run_checks(data_df)
for name, r in check_results.items():
    flag = "PASS" if r["passed"] else "FAIL"
    print(f"{flag:6s} {name:25s} value={r['value']} hard={r['hard']}")

In [ ]:
EXPERIMENT_NAME = f"week19-de-fraud-pipeline-{STUDENT_ID}"
mlflow.set_experiment(EXPERIMENT_NAME)

with mlflow.start_run(run_name="data-validation") as validation_run:
    mlflow.set_tag("stage",         "validation")
    mlflow.set_tag("source_s3_uri", f"s3://{S3_BUCKET}/{SOURCE_KEY}")
    mlflow.log_param("data_version_id", DATA_VERSION_ID)
    mlflow.log_param("data_timestamp",  DATA_TIMESTAMP)

    any_hard_fail = False
    for name, r in check_results.items():
        if isinstance(r["value"], (int, float)):
            mlflow.log_metric(f"check_{name}_value", float(r["value"]))
        mlflow.log_metric(f"check_{name}_passed", 1.0 if r["passed"] else 0.0)
        if r["hard"] and not r["passed"]:
            any_hard_fail = True

    mlflow.set_tag("validation_passed", str(not any_hard_fail))
    VALIDATION_RUN_ID = validation_run.info.run_id

print(f"Validation run logged: {VALIDATION_RUN_ID}")
if any_hard_fail:
    raise RuntimeError("Hard data quality check failed. Training will NOT proceed.")
print("All hard checks passed. Training is allowed to proceed.")

### Lab 2: Add a custom quality check and log it

Bread Financial's risk team has asked you to add a check that catches a specific upstream bug: occasionally a few rows have descriptions shorter than 10 characters, which produces garbage embeddings later (Week 18 RAG pipeline cares about this). Add that check to the validation run.

**Steps**:

1. Compute the count of rows where `description.str.len() < 10`. Call it `short_desc_count`.
2. Decide a threshold: this check fails if `short_desc_count / total > 0.005` (more than 0.5 percent of rows have too-short descriptions).
3. Start a NEW MLflow run named `data-validation-extended`. Log the same `data_version_id` parameter, plus your new metric `short_desc_ratio`, plus a tag `extends_run` pointing to `VALIDATION_RUN_ID` from the demo.
4. Print pass/fail.

**Stretch**: Make `MAX_NULL_RATE`, `MIN_FRAUD_RATIO`, etc. configurable from a Python dict and log the dict as a JSON artifact attached to the validation run via `mlflow.log_dict`.

**Homework Extension**: Convert the demo checks into a reusable `validate_dataframe(df, checks_config)` function in a Python module. Write 2 unit-test-style asserts demonstrating it correctly raises on bad input.

In [ ]:
# Lab 2: extend the validation run with a description-length check

short_desc_count = None  # YOUR CODE
short_desc_ratio = None  # YOUR CODE
EXTENDED_RUN_ID  = None  # YOUR CODE

# SAFETY-NET for Lab 2
if EXTENDED_RUN_ID is None:
    print("Using Lab 2 safety-net.")
    total_rows       = len(data_df)
    short_desc_count = int((data_df["narrative"].astype(str).str.len() < 10).sum())
    short_desc_ratio = short_desc_count / total_rows if total_rows > 0 else 0.0
    with mlflow.start_run(run_name="data-validation-extended") as r:
        mlflow.set_tag("stage",        "validation")
        mlflow.set_tag("extends_run",  VALIDATION_RUN_ID)
        mlflow.log_param("data_version_id", DATA_VERSION_ID)
        mlflow.log_metric("short_desc_ratio", short_desc_ratio)
        mlflow.log_metric("check_short_desc_passed", 1.0 if short_desc_ratio <= 0.005 else 0.0)
        EXTENDED_RUN_ID = r.info.run_id
    print(f"Extended validation run: {EXTENDED_RUN_ID}, ratio={short_desc_ratio:.4f}")

## Topic 3: Feature pipeline as a parent MLflow run, training as a child run

So far you have a validation run. Now you need to capture the feature pipeline: which columns you selected, which filters you applied, which rows survived, and what the output schema looks like. When the training job eventually runs, you want it to appear as a CHILD run under the feature pipeline so the lineage view shows:

```
parent: feature-pipeline (data_version_id=..., output_rows=12000)
  child: training-job (run_id=..., f1=0.91)
```

MLflow supports this natively with `mlflow.start_run(nested=True)` inside another `start_run` context. The local file-store backend on SageMaker Studio handles nested runs the same way the managed tracking server would, so the lineage you build here is portable: swap the tracking URI for a managed MLflow ARN and the same code works.

The parent run logs the pipeline; the child run logs the model. Both inherit the experiment, so the UI groups them together.

In [ ]:
# Apply the feature pipeline. Each step is independently auditable.
step1 = data_df.rename(columns={"narrative": "text", "is_fraud": "label"})[["text", "label"]]
step2 = step1[step1["text"].astype(str).str.len() >= 10]
step3 = step2.dropna(subset=["text", "label"]).reset_index(drop=True)
step3["label"] = step3["label"].astype(int)

PIPELINE_PARAMS = {
    "data_version_id":   DATA_VERSION_ID,
    "validation_run_id": VALIDATION_RUN_ID,
    "step1_rename":      "narrative->text, is_fraud->label",
    "step2_filter":      "len(text) >= 10",
    "step3_dropna":      "text, label",
}
PIPELINE_METRICS = {
    "rows_in":           len(data_df),
    "rows_after_step1":  len(step1),
    "rows_after_step2":  len(step2),
    "rows_after_step3":  len(step3),
}

with mlflow.start_run(run_name="feature-pipeline") as parent_run:
    mlflow.set_tag("stage",             "feature_pipeline")
    mlflow.set_tag("source_s3_uri",     f"s3://{S3_BUCKET}/{SOURCE_KEY}")
    mlflow.set_tag("validation_run_id", VALIDATION_RUN_ID)
    mlflow.log_params(PIPELINE_PARAMS)
    mlflow.log_metrics(PIPELINE_METRICS)
    # Log the output schema as a JSON artifact
    schema_json = json.dumps([{"name": c, "type": str(step3[c].dtype)} for c in step3.columns])
    with open("/tmp/output_schema.json", "w") as f:
        f.write(schema_json)
    mlflow.log_artifact("/tmp/output_schema.json")
    PARENT_RUN_ID = parent_run.info.run_id

print(f"Feature pipeline parent run: {PARENT_RUN_ID}")
print(f"Final row count: {PIPELINE_METRICS['rows_after_step3']}")

In [ ]:
# In the main Week 19 notebook, the actual training job is submitted to SageMaker
# and takes 5-10 minutes. For the DE variant we do not retrain - we re-log the
# instructor's pre-run training metrics AS A CHILD of the feature pipeline parent
# so the lineage is intact.

PRETRAINED_METRICS = {
    "accuracy": 0.94, "precision": 0.91, "recall": 0.88, "f1": 0.895, "eval_loss": 0.18,
}
PRETRAINED_PARAMS = {
    "model_name": "facebook/bart-large-mnli",
    "task":       "zero-shot-classification",
}

# Re-open the parent so the child nests correctly under it
with mlflow.start_run(run_id=PARENT_RUN_ID):
    with mlflow.start_run(run_name="training-job", nested=True) as child_run:
        mlflow.set_tag("stage",             "training")
        mlflow.set_tag("parent_run_id",     PARENT_RUN_ID)
        mlflow.set_tag("validation_run_id", VALIDATION_RUN_ID)
        mlflow.log_params(PRETRAINED_PARAMS)
        mlflow.log_metrics(PRETRAINED_METRICS)
        TRAINING_RUN_ID = child_run.info.run_id

print(f"Training child run: {TRAINING_RUN_ID}")
print(f"  parent:     {PARENT_RUN_ID}")
print(f"  validation: {VALIDATION_RUN_ID}")

### Lab 3: Add a second child run for evaluation metrics

A real training pipeline often has TWO downstream consumers of the same feature pipeline: the training job and a separate offline evaluation job (think holdout test set, fairness checks). Both should nest under the same feature pipeline parent so the lineage stays clean.

**Steps**:

1. Re-open `PARENT_RUN_ID` with `mlflow.start_run(run_id=PARENT_RUN_ID)`.
2. Inside it, start a nested run named `offline-eval` with tag `stage=evaluation`.
3. Log these fake holdout metrics: `holdout_f1=0.88`, `holdout_precision_class1=0.86`, `disparity_ratio=1.04`.
4. Store the eval child run id as `EVAL_RUN_ID`.

**Stretch**: Add a third child run named `data-drift` that logs `drift_score=0.07` and a tag `gate=pass` (pretend the drift detector you built last quarter says everything is fine).

**Homework Extension**: Read the MLflow docs on `mlflow.get_parent_run` and `search_runs(filter_string="tags.mlflow.parentRunId = '<parent_id>'")`. Write a one-liner that lists all child runs of `PARENT_RUN_ID` in a pandas DataFrame.

In [ ]:
# Lab 3: add an offline-eval child run nested under PARENT_RUN_ID

EVAL_RUN_ID = None  # YOUR CODE

# SAFETY-NET for Lab 3
if EVAL_RUN_ID is None:
    print("Using Lab 3 safety-net.")
    with mlflow.start_run(run_id=PARENT_RUN_ID):
        with mlflow.start_run(run_name="offline-eval", nested=True) as r:
            mlflow.set_tag("stage",         "evaluation")
            mlflow.set_tag("parent_run_id", PARENT_RUN_ID)
            mlflow.log_metrics({
                "holdout_f1":               0.88,
                "holdout_precision_class1": 0.86,
                "disparity_ratio":          1.04,
            })
            EVAL_RUN_ID = r.info.run_id
    print(f"Eval child run: {EVAL_RUN_ID}")

## Topic 4: Write a versioned training Parquet snapshot, register the lineage

The standard Week 19 notebook stages CSV to S3. CSV is fine for a demo but it is the wrong format for production: no schema enforcement, expensive parsing, no column pruning. The DE variant uses Parquet.

On SageMaker, the DE pattern is:

1. Write the cleaned feature DataFrame to a NEW versioned S3 prefix: `s3://bread-academy-week19-shared/students/<student_id>/fraud_train/v<N>/`. The `v<N>` segment is a poor-man's catalog version (incremented per dataset cut). This is your durable, queryable training snapshot - independent of the source CSV's S3 VersionId, so even if the raw CSV is lifecycle-expired, your trained-on bytes survive.
2. Inside that prefix write `train.parquet` and `test.parquet`. The schema is enforced by Parquet itself.
3. Tag the resulting SageMaker model package with `parent_run_id`, the source `data_version_id`, and the Parquet snapshot prefix so anyone inspecting the model in the registry can click through to the full lineage.

The Parquet files inherit the dtypes from pandas, so the SageMaker training script can use `pd.read_parquet(...)` instead of `pd.read_csv(...)` and skip all the dtype guessing.

Production note: in a real pipeline this Parquet write would run as a **SageMaker Processing Job** (so it scales beyond what fits in the kernel's memory) or via **AWS Glue** if you need Spark. We do it inline here because the dataset is small and the class focuses on lineage, not scale.

In [ ]:
# Determine a dataset version number for the snapshot prefix.
# Strategy: list existing v<N> subprefixes under students/<id>/fraud_train/ and pick next.
def next_snapshot_version(bucket: str, prefix_root: str) -> int:
    resp = s3_client.list_objects_v2(Bucket=bucket, Prefix=prefix_root, Delimiter="/")
    existing = []
    for cp in resp.get("CommonPrefixes", []):
        leaf = cp["Prefix"].rstrip("/").split("/")[-1]   # e.g. "v3"
        if leaf.startswith("v") and leaf[1:].isdigit():
            existing.append(int(leaf[1:]))
    return (max(existing) + 1) if existing else 1

PREFIX_ROOT  = f"{S3_PREFIX}/fraud_train/"
SNAPSHOT_V   = next_snapshot_version(S3_BUCKET, PREFIX_ROOT)
SNAPSHOT_URI = f"s3://{S3_BUCKET}/{S3_PREFIX}/fraud_train/v{SNAPSHOT_V}"

# Step 1: split
train_df = step3.sample(frac=0.8, random_state=42)
test_df  = step3.drop(train_df.index).reset_index(drop=True)
train_df = train_df.reset_index(drop=True)

# Step 2: write Parquet directly to S3 via awswrangler (handles multipart, content-type, etc.)
TRAIN_S3 = f"{SNAPSHOT_URI}/train.parquet"
TEST_S3  = f"{SNAPSHOT_URI}/test.parquet"
wr.s3.to_parquet(df=train_df, path=TRAIN_S3, boto3_session=boto_session, index=False)
wr.s3.to_parquet(df=test_df,  path=TEST_S3,  boto3_session=boto_session, index=False)

# Step 3: capture the VersionIds of the snapshot objects so the model package
# can pin to the EXACT bytes (defense in depth even though we control the prefix).
train_head = s3_client.head_object(Bucket=S3_BUCKET, Key=f"{S3_PREFIX}/fraud_train/v{SNAPSHOT_V}/train.parquet")
test_head  = s3_client.head_object(Bucket=S3_BUCKET, Key=f"{S3_PREFIX}/fraud_train/v{SNAPSHOT_V}/test.parquet")
TRAIN_PARQUET_VID = train_head.get("VersionId", "null")
TEST_PARQUET_VID  = test_head.get("VersionId",  "null")

# Step 4: log the staging locations as params on the parent run
with mlflow.start_run(run_id=PARENT_RUN_ID):
    mlflow.log_param("snapshot_version", f"v{SNAPSHOT_V}")
    mlflow.log_param("snapshot_uri",     SNAPSHOT_URI)
    mlflow.log_param("train_parquet",    TRAIN_S3)
    mlflow.log_param("test_parquet",     TEST_S3)
    mlflow.log_param("train_parquet_version_id", TRAIN_PARQUET_VID)

print(f"Snapshot v{SNAPSHOT_V} written:")
print(f"  {TRAIN_S3} (VersionId={TRAIN_PARQUET_VID})")
print(f"  {TEST_S3}  (VersionId={TEST_PARQUET_VID})")

In [ ]:
# We will NOT submit a new SageMaker Training Job here (the main Week 19 notebook does that
# and class time is tight). Instead, we register the instructor's pre-run model artifact
# but ATTACH our parent_run_id + source data_version_id + snapshot uri as metadata,
# so the registry knows which feature pipeline produced this model.

PRETRAINED_MODEL_S3 = f"s3://{S3_BUCKET}/pretrained/model.tar.gz"
PACKAGE_GROUP       = f"fraud-classifier-week19-de-{STUDENT_ID}"

# Create group if it does not exist. Use the modeled ResourceInUse exception
# instead of substring-matching the error message (more robust on re-runs).
try:
    sagemaker_client.create_model_package_group(
        ModelPackageGroupName=PACKAGE_GROUP,
        ModelPackageGroupDescription="BART-MNLI zero-shot fraud classifier - Week 19 DE variant (SageMaker)",
    )
    print(f"Created group: {PACKAGE_GROUP}")
except ClientError as e:
    # SageMaker raises ValidationException with "already exists" when the group is reused.
    # Some SDK versions surface a modeled ResourceInUse on the same path.
    code = e.response.get("Error", {}).get("Code", "")
    msg  = str(e)
    if code in ("ResourceInUse", "ValidationException") and ("already exists" in msg or "already existing" in msg):
        print(f"Group {PACKAGE_GROUP} already exists - reusing.")
    else:
        raise

# Region-aware HuggingFace inference image. The ECR account id and region BOTH
# differ per AWS region, so we use the SageMaker SDK's image_uris resolver
# instead of hardcoding us-east-1.
HF_INFERENCE_IMAGE = sagemaker.image_uris.retrieve(
    framework="huggingface",
    region=AWS_REGION,
    version="4.37.0",
    base_framework_version="pytorch2.1.0",
    image_scope="inference",
    instance_type="ml.m5.large",
)
print(f"HF inference image: {HF_INFERENCE_IMAGE}")

resp = sagemaker_client.create_model_package(
    ModelPackageGroupName=PACKAGE_GROUP,
    ModelPackageDescription=f"Lineage: parent_run={PARENT_RUN_ID}, data_version_id={DATA_VERSION_ID}",
    InferenceSpecification={
        "Containers": [{
            "Image":         HF_INFERENCE_IMAGE,
            "ModelDataUrl":  PRETRAINED_MODEL_S3,
            "Environment":   {"HF_TASK": "zero-shot-classification"},
        }],
        "SupportedContentTypes":                ["application/json"],
        "SupportedResponseMIMETypes":           ["application/json"],
        "SupportedRealtimeInferenceInstanceTypes": ["ml.m5.large"],
    },
    ModelApprovalStatus="PendingManualApproval",
    CustomerMetadataProperties={
        "mlflow_parent_run_id":     PARENT_RUN_ID,
        "mlflow_training_run_id":   TRAINING_RUN_ID,
        "mlflow_validation_run_id": VALIDATION_RUN_ID,
        "source_s3_uri":            f"s3://{S3_BUCKET}/{SOURCE_KEY}",
        "source_version_id":        DATA_VERSION_ID,
        "snapshot_uri":             SNAPSHOT_URI,
        "snapshot_version":         f"v{SNAPSHOT_V}",
    },
)
MODEL_PACKAGE_ARN = resp["ModelPackageArn"]
print(f"Registered with lineage tags: {MODEL_PACKAGE_ARN}")

### Lab 4: Query the lineage end-to-end

Pretend you are an auditor. Given only the Model Package ARN, can you walk all the way back to the exact bytes that trained the model? Write the query chain.

**Steps**:

1. Call `sagemaker_client.describe_model_package(ModelPackageName=MODEL_PACKAGE_ARN)` and pull the `CustomerMetadataProperties`.
2. From those properties, extract `mlflow_parent_run_id`, `source_s3_uri`, and `source_version_id`.
3. Use `mlflow.get_run(<parent_run_id>)` to fetch the parent run and print its params and tags.
4. Use `mlflow.search_runs(filter_string="tags.mlflow.parentRunId = '<id>'")` to list every child run under the parent.
5. Use `s3_client.head_object(Bucket=..., Key=..., VersionId=<source_version_id>)` to confirm the exact training bytes are still readable. Report `ContentLength` and `LastModified`.

**Stretch**: Wrap the whole walk into a function `audit_model_package(arn) -> dict` that returns a single dictionary summarizing everything an auditor needs.

**Homework Extension**: Reverse the walk - given a source `VersionId`, list every model package in the registry whose `CustomerMetadataProperties.source_version_id` matches it. This is the "what models did THIS bad batch of data infect?" query.

In [ ]:
# Lab 4 starter

audit = None  # YOUR CODE

# SAFETY-NET for Lab 4
if audit is None:
    print("Using Lab 4 safety-net.")
    desc = sagemaker_client.describe_model_package(ModelPackageName=MODEL_PACKAGE_ARN)
    meta = desc.get("CustomerMetadataProperties", {})
    parent_id    = meta.get("mlflow_parent_run_id")
    source_uri   = meta.get("source_s3_uri")
    source_vid   = meta.get("source_version_id")

    parent = mlflow.get_run(parent_id) if parent_id else None
    children = mlflow.search_runs(
        experiment_names=[EXPERIMENT_NAME],
        filter_string=f"tags.mlflow.parentRunId = '{parent_id}'",
    ) if parent_id else pd.DataFrame()

    # Re-resolve the source bytes by VersionId. Guard against missing metadata
    # (older model packages may not carry source_s3_uri / source_version_id).
    src_size = None
    src_last = None
    if source_uri and source_vid:
        bucket, key = source_uri.replace("s3://", "").split("/", 1)
        src_head = s3_client.head_object(Bucket=bucket, Key=key, VersionId=source_vid)
        src_size = src_head["ContentLength"]
        src_last = str(src_head["LastModified"])
    else:
        print("WARNING: source_s3_uri or source_version_id missing from model package metadata; skipping bytes re-resolution.")

    audit = {
        "model_package_arn":   MODEL_PACKAGE_ARN,
        "parent_run_id":       parent_id,
        "source_s3_uri":       source_uri,
        "source_version_id":   source_vid,
        "source_size_bytes":   src_size,
        "source_last_modified": src_last,
        "child_runs":          [] if children.empty else children["run_id"].tolist(),
        "parent_params":       dict(parent.data.params) if parent else {},
    }
    print(json.dumps(audit, indent=2, default=str))

## Recap

You did the data engineer's half of MLOps, SageMaker-native:

1. **Pinned** the source CSV to a specific S3 `VersionId` (the SageMaker-native equivalent of Delta time travel).
2. **Validated** that pinned data with pandas assertion checks and logged a data-validation run to MLflow as a hard gate.
3. **Structured** the feature pipeline as a parent MLflow run with the training job and offline eval as child runs - giving you a queryable lineage tree.
4. **Wrote** a versioned Parquet snapshot to a student-scoped S3 prefix (`students/<id>/fraud_train/v<N>/`), and registered the model package with `CustomerMetadataProperties` linking back to the parent run, the source S3 VersionId, and the snapshot prefix.

Combined with the ML engineer's half (the main Week 19 SageMaker notebook), Bread Financial now has end-to-end lineage: from raw bytes in `s3://bread-academy-week19-shared/raw/fraud_transactions.csv` to a registered SageMaker model package, with quality gates in the middle and an audit trail at every step. The Week 18 RAG pipeline (KB `FARSQGTONR`, Cohere reranker, Haiku 3 LLM) benefits from the same upstream discipline. Next week (Week 20) you wrap this in CI/CD so the whole pipeline runs on a schedule with DVC + GitHub Actions.

## Homework (async)

1. **Add a "schema drift" check**: write a check that compares the current pandas dtypes to a saved baseline JSON. Fail loud if a column was added, removed, or its type changed. Log it as a new check in the validation run.
2. **Build a "training data" promotion gate**: write a function `promote_training_snapshot(source_version_id)` that copies a specific S3 VersionId of `raw/fraud_transactions.csv` to `promoted/fraud_train_promoted_v<N>.csv` AND writes a sidecar JSON `promoted/fraud_train_promoted_v<N>.json` recording `promoted_from_version_id` so anyone reading the new object knows its origin.
3. **Read** the AWS docs on S3 Lifecycle and noncurrent version expiration. Write 4 sentences on the tradeoff: short retention = cheap storage, long retention = better reproducibility. What value would you pick for fraud training data and why?
4. **Bonus**: explore the MLflow `mlflow.data` API for dataset logging. Re-log the feature pipeline using `mlflow.log_input(dataset)` instead of free-form params, and compare the UX in the MLflow UI.

## Further reading

- S3 object versioning: https://docs.aws.amazon.com/AmazonS3/latest/userguide/Versioning.html
- S3 GetObject with VersionId: https://docs.aws.amazon.com/AmazonS3/latest/API/API_GetObject.html
- awswrangler s3.read_csv / to_parquet: https://aws-sdk-pandas.readthedocs.io/en/stable/api.html
- AWS Glue Data Quality (DQDL): https://docs.aws.amazon.com/glue/latest/dg/glue-data-quality.html
- MLflow nested runs (parent-child): https://mlflow.org/docs/latest/ml/traditional-ml/tutorials/hyperparameter-tuning/part1-child-runs/
- SageMaker Model Registry CustomerMetadataProperties: https://docs.aws.amazon.com/sagemaker/latest/APIReference/API_CreateModelPackage.html
- SageMaker Processing Jobs (production replacement for the inline pandas transforms): https://docs.aws.amazon.com/sagemaker/latest/dg/processing-job.html